# 🧬 NEMPA-Boltz: Análise Exploratória & CRISP-DM
### Branch `rafael` · NEMPA-UFBA/nempa-boltz

**Objetivo:** Análise exploratória completa do `manifest.json` filtrado (complexos MHC–Peptídeo), seguindo o framework **CRISP-DM**, com auditoria das cadeias, inspeção dos arquivos `.npz` e validação do pipeline de filtragem.

---

| Item | Detalhe |
|---|---|
| **Repositório** | [NEMPA-UFBA/nempa-boltz · branch rafael](https://github.com/NEMPA-UFBA/nempa-boltz/tree/rafael) |
| **Dataset fonte** | `boltz1.s3.us-east-2.amazonaws.com/rcsb_processed_targets.tar` |
| **Padrão de dados** | RCSB PDB / PDBx-mmCIF / wwPDB |
| **Modelo base** | Boltz-1 (MIT License) — jwohlwend/boltz |
| **Foco biológico** | Complexos MHC Classe I – Peptídeo Antigênico |

---

## Índice
1. [Fase 1 — Entendimento do Negócio (CRISP-DM)](#fase1)
2. [Fase 2 — Entendimento dos Dados](#fase2)
3. [Fase 3 — Preparação dos Dados](#fase3)
4. [Fase 4 — EDA: manifest.json filtrado](#fase4)
5. [Fase 5 — EDA: Cadeias (chains)](#fase5)
6. [Fase 6 — EDA: Interfaces](#fase6)
7. [Fase 7 — Auditoria do Pipeline (scripts NEMPA)](#fase7)
8. [Fase 8 — Inspeção de Arquivos .npz](#fase8)
9. [Fase 9 — Validação dos Critérios Biológicos](#fase9)
10. [Fase 10 — Relatório CRISP-DM Final](#fase10)


## <a id='fase1'></a>Fase 1 — Entendimento do Negócio (CRISP-DM)

O **Complexo de Histocompatibilidade Principal (MHC) Classe I** apresenta peptídeos antigênicos na superfície celular para reconhecimento por células T citotóxicas (CD8+). Compreender a estrutura desses complexos é essencial para:
- **Vacinas personalizadas** (neoantígenos tumorais)
- **Imunoterapia** (predição de epitopos)
- **Doenças autoimunes** (reconhecimento de self × non-self)

### Objetivo analítico
> Verificar a **qualidade, completude e consistência** do `manifest.json` filtrado para treino do Boltz no contexto MHC–Peptídeo, identificando quantos complexos atendem aos critérios biológicos e onde estão os pontos de perda do pipeline.

### Critérios biológicos de negócio
| Critério | Regra | Justificativa |
|---|---|---|
| MHC Completo | `num_residues ≥ 200` | MHC Classe I tem ~270–280 resíduos; valores menores indicam estrutura truncada |
| Peptídeo Presente | `num_residues ≤ 25` | Peptídeos MHC-I têm tipicamente 8–11 resíduos; máximo biológico ~25 |
| Pelo menos 2 cadeias | `len(chains) ≥ 2` | Complexo mínimo = 1 MHC + 1 Peptídeo |
| Interface válida | `interface.valid == True` | Contato direto MHC–Peptídeo confirmado |


## <a id='fase2'></a>Fase 2 — Entendimento dos Dados
### 2.1 Instalação e Imports

In [ ]:
# Instalar dependências se necessário
# !pip install pandas numpy matplotlib seaborn plotly

import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Configurações visuais
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 130, 'figure.figsize': (10, 5),
                     'axes.titlesize': 13, 'axes.labelsize': 11})

print("✅ Imports OK")
print(f"Pandas: {pd.__version__} | NumPy: {np.__version__}")


: 

### 2.2 Caminhos do Projeto

In [ ]:
# ─────────────────────────────────────────────────────────
#  Ajuste os caminhos conforme seu ambiente local ou CMCAD
# ─────────────────────────────────────────────────────────
BASE_DIR          = Path(".")
RCSB_DIR          = BASE_DIR / "rcsb_processed_targets"
MHC_DATA_DIR      = BASE_DIR / "mhc_data"
DATASET_LIMPO_DIR = MHC_DATA_DIR / "dataset_limpo"

# Manifestos
MANIFEST_MASTER   = RCSB_DIR / "manifest.json"
MANIFEST_FILTERED = MHC_DATA_DIR / "mhc_manifest_filtered.json"
MANIFEST_LIMPO    = DATASET_LIMPO_DIR / "manifest.json"

# Diretórios de estruturas e MSA
STRUCTURES_ORIG   = RCSB_DIR / "structures"
STRUCTURES_LIMPO  = DATASET_LIMPO_DIR / "structures"
MSA_DIR           = DATASET_LIMPO_DIR / "msa"

# CSV clínico
CSV_TCR3D         = MHC_DATA_DIR / "TCR3d_data.csv"

# Validação
VALIDATION_IDS    = BASE_DIR / "validation_ids.txt"

print("Caminhos configurados:")
for nome, path in [
    ("manifest master",   MANIFEST_MASTER),
    ("manifest filtrado", MANIFEST_FILTERED),
    ("manifest limpo",    MANIFEST_LIMPO),
    ("structures (orig)", STRUCTURES_ORIG),
    ("structures (limpo)",STRUCTURES_LIMPO),
    ("MSA (limpo)",       MSA_DIR),
    ("CSV TCR3D",         CSV_TCR3D),
]:
    existe = "✅" if path.exists() else "❌ não encontrado"
    print(f"  {nome:<22} → {path}  {existe}")


## <a id='fase3'></a>Fase 3 — Preparação dos Dados
### 3.1 Carregamento do Manifest Filtrado

In [ ]:
def carregar_manifest(path: Path) -> list:
    """Carrega um manifest.json e retorna a lista de records."""
    if not path.exists():
        print(f"⚠️  Arquivo não encontrado: {path}")
        return []
    with open(path, "r") as f:
        data = json.load(f)
    # suporte a formato list direto ou dict com chave 'records'
    if isinstance(data, list):
        return data
    elif isinstance(data, dict) and "records" in data:
        return data["records"]
    return []

# Carregar manifestos disponíveis
manifest_filtered = carregar_manifest(MANIFEST_FILTERED)
manifest_limpo    = carregar_manifest(MANIFEST_LIMPO)

# Usar o melhor disponível para EDA principal
manifest = manifest_limpo if manifest_limpo else manifest_filtered

print(f"Manifest filtrado (mhc_manifest_filtered.json) : {len(manifest_filtered):>6} registros")
print(f"Manifest limpo   (dataset_limpo/manifest.json) : {len(manifest_limpo):>6} registros")
print(f"\n→ EDA principal usando: {len(manifest)} registros")


### 3.2 Normalização para DataFrames

In [ ]:
def manifest_to_dataframes(records: list):
    """
    Normaliza o manifest.json em 3 DataFrames relacionais:
      - df_entries  : um registro por entrada PDB (nível structure)
      - df_chains   : um registro por cadeia
      - df_ifaces   : um registro por interface
    """
    entries, chains, ifaces = [], [], []

    for rec in records:
        pid = rec.get("id", "")
        s   = rec.get("structure", {}) or {}

        entries.append({
            "id"            : pid,
            "resolution"    : s.get("resolution"),
            "method"        : s.get("method"),
            "deposited"     : pd.to_datetime(s.get("deposited"), errors="coerce"),
            "released"      : pd.to_datetime(s.get("released"),  errors="coerce"),
            "revised"       : pd.to_datetime(s.get("revised"),   errors="coerce"),
            "num_chains"    : s.get("num_chains"),
            "num_interfaces": s.get("num_interfaces"),
            "has_affinity"  : rec.get("affinity") is not None,
            "has_md"        : rec.get("md") is not None,
        })

        for ch in rec.get("chains", []):
            chains.append({
                "pdb_id"      : pid,
                "chain_id"    : ch.get("chain_id"),
                "chain_name"  : ch.get("chain_name"),
                "mol_type"    : ch.get("mol_type"),
                "cluster_id"  : str(ch.get("cluster_id", "")),
                "msa_id"      : str(ch.get("msa_id", "")),
                "template_id" : str(ch.get("template_id", "")),
                "num_residues": ch.get("num_residues"),
                "valid"       : ch.get("valid"),
            })

        for ifc in rec.get("interfaces", []):
            ifaces.append({
                "pdb_id" : pid,
                "chain_1": ifc.get("chain_1"),
                "chain_2": ifc.get("chain_2"),
                "valid"  : ifc.get("valid"),
                # num_contacts ausente no NEMPA — tratar graciosamente
                "num_contacts": ifc.get("num_contacts"),
            })

    df_e = pd.DataFrame(entries)
    df_c = pd.DataFrame(chains)
    df_i = pd.DataFrame(ifaces)

    # Enum mol_type → rótulo
    mol_map = {0: "Proteína", 1: "RNA", 2: "DNA", 3: "Ligante"}
    if not df_c.empty:
        df_c["mol_type_label"] = df_c["mol_type"].map(mol_map).fillna("Desconhecido")

    return df_e, df_c, df_i

df_entries, df_chains, df_ifaces = manifest_to_dataframes(manifest)

print(f"df_entries : {df_entries.shape}")
print(f"df_chains  : {df_chains.shape}")
print(f"df_ifaces  : {df_ifaces.shape}")


In [ ]:
# Visão geral dos DataFrames
print("── df_entries ──")
display(df_entries.head(3))
print("\n── df_chains ──")
display(df_chains.head(6))
print("\n── df_ifaces ──")
display(df_ifaces.head(4))


## <a id='fase4'></a>Fase 4 — EDA: Entradas PDB (manifest.json)
### 4.1 Estatísticas Gerais

In [ ]:
print("=" * 55)
print("  SUMÁRIO DO MANIFEST FILTRADO")
print("=" * 55)
print(f"  Total de entradas PDB          : {len(df_entries)}")
print(f"  IDs únicos                     : {df_entries['id'].nunique()}")
print(f"  Com dado de afinidade          : {df_entries['has_affinity'].sum()}")
print(f"  Com dado de MD                 : {df_entries['has_md'].sum()}")
print(f"  Resolução média (Å)            : {df_entries['resolution'].replace(0, np.nan).mean():.2f}")
print(f"  Resolução mediana (Å)          : {df_entries['resolution'].replace(0, np.nan).median():.2f}")
print(f"  Num_chains — média             : {df_entries['num_chains'].mean():.1f}")
print(f"  Num_chains — máximo            : {df_entries['num_chains'].max()}")
print("=" * 55)
print("\nDistribuição de métodos experimentais:")
print(df_entries['method'].value_counts().to_string())


### 4.2 Resolução — Distribuição

In [ ]:
res = df_entries['resolution'].replace(0, np.nan).dropna()
res_filtrada = res[res < 10]  # remover outliers extremos

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histograma
axes[0].hist(res_filtrada, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(res_filtrada.median(), color='tomato', linestyle='--', lw=2,
                label=f'Mediana: {res_filtrada.median():.2f} Å')
axes[0].axvline(res_filtrada.mean(),   color='orange',  linestyle='--', lw=2,
                label=f'Média: {res_filtrada.mean():.2f} Å')
axes[0].axvspan(0, 2.5, alpha=0.08, color='green', label='Alta resolução (< 2.5 Å)')
axes[0].set_title("Distribuição de Resolução (Å)")
axes[0].set_xlabel("Resolução (Å)")
axes[0].set_ylabel("Contagem")
axes[0].legend(fontsize=9)

# Box
axes[1].boxplot(res_filtrada, vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6),
                medianprops=dict(color='tomato', lw=2))
axes[1].set_title("Box-plot — Resolução (Å)")
axes[1].set_ylabel("Resolução (Å)")

plt.tight_layout()
plt.savefig("output/eda_01_resolucao.png", bbox_inches='tight')
plt.show()
print(f"Cobertura: {len(res_filtrada)}/{len(df_entries)} entradas com resolução válida")


### 4.3 Método Experimental

In [ ]:
method_counts = df_entries['method'].fillna('não informado').value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(method_counts.index, method_counts.values,
               color=sns.color_palette("muted", len(method_counts)))
ax.bar_label(bars, padding=4, fontsize=10)
ax.set_title("Método Experimental de Determinação Estrutural")
ax.set_xlabel("Número de Entradas")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("output/eda_02_metodo.png", bbox_inches='tight')
plt.show()


### 4.4 Evolução Temporal — Deposição

In [ ]:
dep = df_entries['deposited'].dropna()
if len(dep) > 0:
    dep_year = dep.dt.year.value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(dep_year.index, dep_year.values, color='teal', alpha=0.8, edgecolor='white')
    ax.set_title("Deposições no PDB por Ano")
    ax.set_xlabel("Ano de Deposição")
    ax.set_ylabel("Número de Estruturas")
    ax.axvline(dep_year.index[dep_year.cumsum() >= dep_year.sum() * 0.5][0],
               color='tomato', linestyle='--', lw=1.5, label='50% acumulado')
    ax.legend()
    plt.tight_layout()
    plt.savefig("output/eda_03_temporal.png", bbox_inches='tight')
    plt.show()
    print(f"Intervalo: {dep.dt.year.min()} – {dep.dt.year.max()}")


### 4.5 Número de Cadeias por Entrada

In [ ]:
nc = df_entries['num_chains'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(nc.index.astype(str), nc.values, color='mediumslateblue', edgecolor='white')
ax.set_title("Distribuição do Número de Cadeias por Entrada PDB")
ax.set_xlabel("num_chains")
ax.set_ylabel("Frequência")
for i, v in enumerate(nc.values):
    ax.text(i, v + 0.2, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig("output/eda_04_num_chains.png", bbox_inches='tight')
plt.show()
print("\nContagem:")
print(nc.to_string())


## <a id='fase5'></a>Fase 5 — EDA: Cadeias (chains)
### 5.1 Visão Geral das Cadeias

In [ ]:
print("=" * 55)
print("  SUMÁRIO DAS CADEIAS")
print("=" * 55)
print(f"  Total de cadeias               : {len(df_chains)}")
print(f"  Cadeias válidas (valid=True)   : {df_chains['valid'].sum()}")
print(f"  Cadeias inválidas              : {(~df_chains['valid']).sum()}")
pct = df_chains['valid'].mean() * 100 if len(df_chains) > 0 else 0
print(f"  Taxa de validade               : {pct:.1f}%")
print(f"  Média de resíduos (válidas)    : {df_chains[df_chains['valid']]['num_residues'].mean():.1f}")
print(f"  Média de resíduos (inválidas)  : {df_chains[~df_chains['valid']]['num_residues'].mean():.1f}")
print("\nPor mol_type:")
print(df_chains.groupby('mol_type_label')['valid'].value_counts().to_string())


### 5.2 Distribuição por Tipo Molecular

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Pizza
mol_ct = df_chains['mol_type_label'].value_counts()
axes[0].pie(mol_ct.values, labels=mol_ct.index,
            autopct='%1.1f%%', startangle=90,
            colors=sns.color_palette("pastel", len(mol_ct)))
axes[0].set_title("Tipos Moleculares — Todas as Cadeias")

# Stacked bar valid/invalid
mol_valid = df_chains.groupby(['mol_type_label', 'valid']).size().unstack(fill_value=0)
mol_valid.plot(kind='bar', ax=axes[1], color=['tomato','steelblue'],
               edgecolor='white', width=0.6)
axes[1].set_title("Válidas vs Inválidas por Tipo Molecular")
axes[1].set_xlabel("Tipo Molecular")
axes[1].set_ylabel("Contagem")
axes[1].legend(['Inválida (False)', 'Válida (True)'], fontsize=9)
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig("output/eda_05_mol_type.png", bbox_inches='tight')
plt.show()


### 5.3 Distribuição de Resíduos — Critério Biológico MHC-I

In [ ]:
# Separar por papel biológico esperado
prot = df_chains[df_chains['mol_type'] == 0].copy()

# Classificação NEMPA baseada em num_residues
def classificar_papel(n):
    if n >= 200:  return "MHC (≥200 res)"
    if n <= 25:   return "Peptídeo (≤25 res)"
    return "β₂m / outro (26–199 res)"

prot['papel'] = prot['num_residues'].apply(classificar_papel)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma por classe
for label, color in [("MHC (≥200 res)", "steelblue"),
                     ("β₂m / outro (26–199 res)", "orange"),
                     ("Peptídeo (≤25 res)", "tomato")]:
    sub = prot[prot['papel'] == label]['num_residues']
    axes[0].hist(sub, bins=30, alpha=0.7, label=f"{label} (n={len(sub)})", color=color)

axes[0].axvline(25,  color='tomato',    linestyle='--', lw=1.5, label='Limite Peptídeo (25)')
axes[0].axvline(200, color='steelblue', linestyle='--', lw=1.5, label='Limite MHC (200)')
axes[0].set_title("Distribuição de Resíduos — Cadeias Proteicas")
axes[0].set_xlabel("num_residues")
axes[0].set_ylabel("Frequência")
axes[0].set_xlim(0, 500)
axes[0].legend(fontsize=8)

# Pizza dos papéis
papel_ct = prot['papel'].value_counts()
axes[1].pie(papel_ct.values, labels=papel_ct.index,
            autopct='%1.1f%%', startangle=90,
            colors=['steelblue', 'orange', 'tomato'])
axes[1].set_title("Papéis Biológicos — Cadeias Proteicas")

plt.tight_layout()
plt.savefig("output/eda_06_residuos_mhc.png", bbox_inches='tight')
plt.show()

print("\nContagem por papel biológico:")
print(prot['papel'].value_counts().to_string())


### 5.4 Tamanho de Peptídeos — Zoom (≤ 25 resíduos)

In [ ]:
pep = prot[prot['num_residues'] <= 25]['num_residues']
pep_ct = pep.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(pep_ct.index, pep_ct.values, color='tomato', edgecolor='white', alpha=0.85)
ax.axvspan(8, 11, alpha=0.15, color='green', label='Comprimento canônico MHC-I (8–11 mers)')
ax.set_title("Comprimento dos Peptídeos Antigênicos (≤ 25 resíduos)")
ax.set_xlabel("Número de Resíduos")
ax.set_ylabel("Frequência")
ax.legend()
plt.tight_layout()
plt.savefig("output/eda_07_peptideo_len.png", bbox_inches='tight')
plt.show()

print(f"Total de peptídeos (≤25 res): {len(pep)}")
print(f"Comprimento mais frequente  : {pep.mode().values[0] if len(pep)>0 else 'N/A'} resíduos")
print(f"Comprimento médio           : {pep.mean():.1f} resíduos")


### 5.5 Cluster IDs — Diversidade de Sequência

In [ ]:
valid_chains = df_chains[df_chains['valid'] == True]

n_clusters   = valid_chains['cluster_id'].nunique()
n_no_cluster = (valid_chains['cluster_id'] == '-1').sum()
n_msas       = valid_chains[valid_chains['msa_id'] != '-1']['msa_id'].nunique()

print(f"Cadeias válidas              : {len(valid_chains)}")
print(f"Clusters únicos (40% sim.)   : {n_clusters}")
print(f"Sem cluster (cluster_id=-1)  : {n_no_cluster}")
print(f"MSAs únicas disponíveis      : {n_msas}")
print(f"\nTop 10 cluster_ids mais frequentes:")
print(valid_chains['cluster_id'].value_counts().head(10).to_string())


## <a id='fase6'></a>Fase 6 — EDA: Interfaces
### 6.1 Visão Geral das Interfaces

In [ ]:
print("=" * 55)
print("  SUMÁRIO DAS INTERFACES")
print("=" * 55)
print(f"  Total de interfaces            : {len(df_ifaces)}")
if len(df_ifaces) > 0:
    print(f"  Interfaces válidas             : {df_ifaces['valid'].sum()}")
    print(f"  Interfaces inválidas           : {(~df_ifaces['valid']).sum()}")
    pct_if = df_ifaces['valid'].mean() * 100
    print(f"  Taxa de validade               : {pct_if:.1f}%")
    if 'num_contacts' in df_ifaces.columns and df_ifaces['num_contacts'].notna().any():
        print(f"  Contatos médios (válidas)      : {df_ifaces[df_ifaces['valid']]['num_contacts'].mean():.1f}")
    else:
        print("  num_contacts                   : ausente no NEMPA (campo removido)")


### 6.2 Interfaces Válidas vs Inválidas

In [ ]:
if len(df_ifaces) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Pizza
    val_ct = df_ifaces['valid'].value_counts()
    labels = [f"Válida\n(n={val_ct.get(True,0)})", f"Inválida\n(n={val_ct.get(False,0)})"]
    axes[0].pie([val_ct.get(True,0), val_ct.get(False,0)],
                labels=labels, autopct='%1.1f%%',
                colors=['steelblue','tomato'], startangle=90)
    axes[0].set_title("Interfaces: Válidas vs Inválidas")

    # Interfaces por entrada PDB
    if_por_pdb = df_ifaces.groupby('pdb_id')['valid'].sum()
    axes[1].hist(if_por_pdb.values, bins=range(0, int(if_por_pdb.max())+2),
                 color='steelblue', edgecolor='white', alpha=0.85)
    axes[1].set_title("Interfaces Válidas por Entrada PDB")
    axes[1].set_xlabel("Número de Interfaces Válidas")
    axes[1].set_ylabel("Frequência")

    plt.tight_layout()
    plt.savefig("output/eda_08_interfaces.png", bbox_inches='tight')
    plt.show()


## <a id='fase7'></a>Fase 7 — Auditoria do Pipeline NEMPA
### 7.1 Função: `filter_mhc.py` — Filtro por CSV TCR3d

In [ ]:
def simular_filter_mhc(manifest_master_path: Path, csv_path: Path) -> dict:
    """
    Replica a lógica de filter_mhc.py:
    Filtra o manifest master pelos PDB IDs do CSV TCR3d_data.csv
    """
    resultado = {"csv_ids": 0, "manifesto_ids": 0, "interseção": 0,
                 "ausentes_no_manifesto": [], "ids_filtrados": []}

    if not csv_path.exists():
        print(f"⚠️  CSV não encontrado: {csv_path}")
        return resultado

    df_csv = pd.read_csv(csv_path)
    ids_csv = set(df_csv['PDB ID'].dropna().str.lower().unique())
    resultado['csv_ids'] = len(ids_csv)

    manifest_m = carregar_manifest(manifest_master_path)
    ids_manifesto = {r['id'].lower() for r in manifest_m}
    resultado['manifesto_ids'] = len(ids_manifesto)

    interseção = ids_csv & ids_manifesto
    resultado['interseção'] = len(interseção)
    resultado['ausentes_no_manifesto'] = sorted(ids_csv - ids_manifesto)
    resultado['ids_filtrados'] = sorted(interseção)

    return resultado

res_filtro = simular_filter_mhc(MANIFEST_MASTER, CSV_TCR3D)

print("─" * 50)
print("  RESULTADO DE filter_mhc.py")
print("─" * 50)
print(f"  IDs no CSV TCR3d            : {res_filtro['csv_ids']}")
print(f"  IDs no manifest master      : {res_filtro['manifesto_ids']}")
print(f"  Interseção (prontos)        : {res_filtro['interseção']}")
print(f"  Ausentes no manifesto       : {len(res_filtro['ausentes_no_manifesto'])}")
if res_filtro['ausentes_no_manifesto']:
    print(f"  Primeiros ausentes          : {res_filtro['ausentes_no_manifesto'][:10]}")


### 7.2 Função: `auditar_dados.py` — Auditoria de Arquivos .npz

In [ ]:
def simular_auditar_dados(manifest_path: Path, structures_dir: Path,
                          csv_path: Path) -> pd.DataFrame:
    """
    Replica a lógica de auditar_dados.py:
    Cruza CSV × manifest × presença física do .npz
    """
    rows = []
    if not csv_path.exists():
        print(f"⚠️  CSV não encontrado: {csv_path}")
        return pd.DataFrame()

    df_csv = pd.read_csv(csv_path)
    ids_csv = set(df_csv['PDB ID'].dropna().str.lower().unique())

    manifest_data = carregar_manifest(manifest_path)
    ids_manifesto = {r['id'].lower() for r in manifest_data}

    for pid in sorted(ids_csv):
        em_manifesto = pid in ids_manifesto
        tem_npz = (structures_dir / f"{pid}.npz").exists() if structures_dir.exists() else False
        status = ("✅ pronto" if (em_manifesto and tem_npz)
                  else "⚠️ sem_npz" if em_manifesto
                  else "❌ não_no_manifesto")
        rows.append({"pdb_id": pid, "em_manifesto": em_manifesto,
                     "tem_npz": tem_npz, "status": status})

    return pd.DataFrame(rows)

df_audit = simular_auditar_dados(MANIFEST_MASTER, STRUCTURES_ORIG, CSV_TCR3D)

if not df_audit.empty:
    print("Resumo da auditoria:")
    print(df_audit['status'].value_counts().to_string())
    print()
    display(df_audit.head(15))


### 7.3 Visualização do Funil de Filtragem

In [ ]:
# Funil CRISP-DM: perdas em cada etapa do pipeline
etapas = []
valores = []

# Etapa 0: CSV
if not df_audit.empty:
    etapas.append("1. CSV TCR3d\n(IDs clínicos)")
    valores.append(len(df_audit))

    # Etapa 1: no manifesto
    no_manifest = df_audit['em_manifesto'].sum()
    etapas.append("2. No manifest\nmaster (Boltz)")
    valores.append(no_manifest)

    # Etapa 2: tem .npz
    tem_npz = df_audit['tem_npz'].sum()
    etapas.append("3. Arquivo .npz\ndisponível")
    valores.append(tem_npz)

# Etapa 3: cadeias válidas
n_entradas_validas = df_chains.groupby('pdb_id')['valid'].any().sum()
etapas.append("4. Com cadeia\nválida (NEMPA)")
valores.append(n_entradas_validas)

# Etapa 4: interface válida
if len(df_ifaces) > 0:
    n_if_validas = df_ifaces[df_ifaces['valid']].groupby('pdb_id').size().gt(0).sum()
    etapas.append("5. Interface\nválida MHC-Pep")
    valores.append(n_if_validas)

if valores:
    cores = sns.color_palette("Blues_d", len(valores))
    fig, ax = plt.subplots(figsize=(len(valores)*2.2, 5))
    bars = ax.bar(range(len(etapas)), valores, color=cores, edgecolor='white', width=0.6)
    ax.bar_label(bars, labels=[f"{v:,}" for v in valores], padding=5, fontsize=11, fontweight='bold')

    for i in range(1, len(valores)):
        pct = valores[i]/valores[i-1]*100 if valores[i-1] > 0 else 0
        ax.annotate(f"↓ {100-pct:.1f}%",
                    xy=((i-0.5), max(valores)*0.5),
                    ha='center', va='center', fontsize=9, color='tomato')

    ax.set_xticks(range(len(etapas)))
    ax.set_xticklabels(etapas, fontsize=9)
    ax.set_title("Funil de Filtragem do Pipeline NEMPA", fontsize=13, fontweight='bold')
    ax.set_ylabel("Número de Entradas PDB")
    ax.set_ylim(0, max(valores) * 1.15)
    plt.tight_layout()
    plt.savefig("output/eda_09_funil.png", bbox_inches='tight')
    plt.show()


### 7.4 Função: `limpar_cadeias.py` — Critério Biológico

In [ ]:
def simular_limpar_cadeias(records: list) -> pd.DataFrame:
    """
    Replica a lógica de limpar_cadeias.py:
    Classifica cada entrada PDB pelos critérios biológicos MHC-I
    """
    rows = []
    for rec in records:
        pid = rec['id']
        chains = rec.get('chains', [])

        if len(chains) < 2:
            rows.append({"pdb_id": pid, "decisao": "❌ Menos de 2 cadeias",
                         "maior": None, "menor": None})
            continue

        prot = [c for c in chains if c.get('mol_type') == 0]
        if not prot:
            rows.append({"pdb_id": pid, "decisao": "❌ Sem cadeia proteica",
                         "maior": None, "menor": None})
            continue

        tamanhos = [c['num_residues'] for c in prot]
        maior = max(tamanhos)
        menor = min(tamanhos)

        motivos = []
        if maior < 200: motivos.append("MHC Incompleto")
        if menor > 25:  motivos.append("Pep. Ausente")

        if motivos:
            rows.append({"pdb_id": pid, "decisao": "❌ " + " + ".join(motivos),
                         "maior": maior, "menor": menor})
        else:
            rows.append({"pdb_id": pid, "decisao": "✅ Válido",
                         "maior": maior, "menor": menor})

    return pd.DataFrame(rows)

df_limpeza = simular_limpar_cadeias(manifest)

print("Resultado da simulação de limpar_cadeias.py:")
print(df_limpeza['decisao'].value_counts().to_string())
print()
print("Detalhes dos descartados:")
display(df_limpeza[df_limpeza['decisao'].str.startswith("❌")].head(20))


### 7.5 Função: `analise_descartes.py` — Diagnóstico Detalhado

In [ ]:
if not df_limpeza.empty:
    decisao_ct = df_limpeza['decisao'].value_counts()

    fig, ax = plt.subplots(figsize=(10, 4))
    cores_diag = ['tomato' if '❌' in d else 'steelblue' for d in decisao_ct.index]
    bars = ax.barh(decisao_ct.index, decisao_ct.values, color=cores_diag, edgecolor='white')
    ax.bar_label(bars, padding=4, fontsize=10)
    ax.set_title("Diagnóstico de Descarte — analise_descartes.py (Simulado)", fontsize=12)
    ax.set_xlabel("Número de Entradas PDB")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig("output/eda_10_descartes.png", bbox_inches='tight')
    plt.show()

    # Scatter: maior × menor para os descartados
    disc = df_limpeza[df_limpeza['decisao'].str.startswith("❌")].dropna()
    if len(disc) > 0:
        fig, ax = plt.subplots(figsize=(9, 5))
        scatter = ax.scatter(disc['maior'], disc['menor'],
                             alpha=0.5, c='tomato', s=30)
        ax.axvline(200, color='steelblue', linestyle='--', lw=1.5, label='Limite MHC (200)')
        ax.axhline(25,  color='orange',    linestyle='--', lw=1.5, label='Limite Peptídeo (25)')
        ax.set_xlabel("Maior cadeia (num_residues)")
        ax.set_ylabel("Menor cadeia (num_residues)")
        ax.set_title("Estruturas Descartadas: Maior × Menor cadeia")
        ax.legend()
        plt.tight_layout()
        plt.savefig("output/eda_11_scatter_descarte.png", bbox_inches='tight')
        plt.show()


## <a id='fase8'></a>Fase 8 — Inspeção de Arquivos .npz
### 8.1 `raio_x_npz.py` — Inspecionar Tensores

In [ ]:
def raio_x_npz(path: Path) -> pd.DataFrame:
    """
    Replica raio_x_npz.py: exibe todos os tensores de um arquivo .npz
    Retorna DataFrame com chave, shape, dtype e estatísticas básicas.
    """
    if not path.exists():
        print(f"⚠️  Arquivo não encontrado: {path}")
        return pd.DataFrame()

    dados = np.load(path, allow_pickle=True)
    rows = []
    print(f"\n{'='*60}")
    print(f"🔬 RAIO-X: {path.name}")
    print(f"{'='*60}")
    for key in dados.files:
        arr = dados[key]
        row = {"chave": key, "shape": arr.shape, "dtype": str(arr.dtype),
               "min": None, "max": None, "mean": None}
        if np.issubdtype(arr.dtype, np.number) and arr.size > 0:
            row.update({"min": float(arr.min()), "max": float(arr.max()),
                        "mean": float(arr.mean())})
        rows.append(row)
        print(f"  📦 {key:<20} | shape: {str(arr.shape):<20} | dtype: {arr.dtype}")
    print(f"{'='*60}\n")
    return pd.DataFrame(rows)

# Testar com 1a1m (original e limpo)
df_npz_orig  = raio_x_npz(STRUCTURES_ORIG  / "1a1m.npz")
df_npz_limpo = raio_x_npz(STRUCTURES_LIMPO / "1a1m.npz")

if not df_npz_orig.empty:
    print("Tensores — arquivo original:")
    display(df_npz_orig)
if not df_npz_limpo.empty:
    print("Tensores — arquivo limpo (NEMPA):")
    display(df_npz_limpo)


### 8.2 `auditoria_mascara.py` — Comparar Máscaras Original × Limpo

In [ ]:
def auditar_mascaras(pdb_id: str, structures_orig: Path, structures_limpo: Path) -> pd.DataFrame:
    """
    Replica auditoria_mascara.py: compara array 'mask' antes e depois da limpeza NEMPA.
    """
    path_orig  = structures_orig  / f"{pdb_id}.npz"
    path_limpo = structures_limpo / f"{pdb_id}.npz"

    if not path_orig.exists() or not path_limpo.exists():
        print(f"⚠️  Arquivos de {pdb_id} não encontrados.")
        return pd.DataFrame()

    orig  = np.load(path_orig,  allow_pickle=True)
    limpo = np.load(path_limpo, allow_pickle=True)

    mask_o = orig['mask']
    mask_l = limpo['mask']

    rows = []
    for i, (mo, ml) in enumerate(zip(mask_o, mask_l)):
        mudou = (mo != ml)
        rows.append({"cadeia_idx": i,
                     "mask_original": bool(mo),
                     "mask_limpa":    bool(ml),
                     "alterado":      mudou,
                     "ação": "🔄 REMOVIDA" if (mo and not ml) else
                             "🔄 ADICIONADA" if (not mo and ml) else
                             "✅ INTACTO"})
    df = pd.DataFrame(rows)

    print(f"\n{'='*50}")
    print(f"📊 AUDITORIA DE MÁSCARA: {pdb_id.upper()}")
    print(f"{'='*50}")
    print(f"Cadeias totais: {len(df)}")
    print(f"Alteradas     : {df['alterado'].sum()}")
    print(f"Mantidas True : {df['mask_limpa'].sum()}")
    print(f"Mantidas False: {(~df['mask_limpa']).sum()}")
    return df

df_mask_1a1m = auditar_mascaras("1a1m", STRUCTURES_ORIG, STRUCTURES_LIMPO)
if not df_mask_1a1m.empty:
    display(df_mask_1a1m)


### 8.3 `auditoria_em_lote.py` — Auditoria de Todas as Estruturas Limpas

In [ ]:
def auditar_em_lote(structures_limpo: Path) -> pd.DataFrame:
    """
    Replica auditoria_em_lote.py: para cada .npz no dataset limpo,
    reporta quais índices de cadeia ficaram com mask=True.
    """
    if not structures_limpo.exists():
        print(f"⚠️  Diretório não encontrado: {structures_limpo}")
        return pd.DataFrame()

    arquivos = sorted(structures_limpo.glob("*.npz"))
    rows = []
    for arq in arquivos:
        pid = arq.stem
        try:
            dados = np.load(arq, allow_pickle=True)
            mask = dados['mask']
            indices_true = [i for i, v in enumerate(mask) if v]
            rows.append({"pdb_id": pid.upper(),
                         "total_cadeias": len(mask),
                         "indices_validos": str(indices_true),
                         "n_validas": len(indices_true),
                         "status": "OK"})
        except Exception as e:
            rows.append({"pdb_id": pid.upper(), "total_cadeias": None,
                         "indices_validos": None, "n_validas": None,
                         "status": f"ERRO: {e}"})

    df = pd.DataFrame(rows)
    print(f"Auditoria de {len(df)} arquivos .npz no dataset limpo")
    print(f"Status OK   : {(df['status']=='OK').sum()}")
    print(f"Com erros   : {(df['status']!='OK').sum()}")
    return df

df_lote = auditar_em_lote(STRUCTURES_LIMPO)
if not df_lote.empty:
    display(df_lote.head(20))

    # Distribuição de cadeias válidas por estrutura
    ok = df_lote[df_lote['status']=='OK']['n_validas']
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(ok, bins=range(0, int(ok.max())+2) if len(ok)>0 else 5,
            color='steelblue', edgecolor='white')
    ax.set_title("Número de Cadeias Válidas por Estrutura (dataset limpo)")
    ax.set_xlabel("Cadeias com mask=True")
    ax.set_ylabel("Frequência")
    plt.tight_layout()
    plt.savefig("output/eda_12_mascaras_lote.png", bbox_inches='tight')
    plt.show()


## <a id='fase9'></a>Fase 9 — Validação dos Critérios Biológicos
### 9.1 Checagem Completa de Consistência

In [ ]:
def validacao_completa(records: list, structures_dir: Path) -> pd.DataFrame:
    """
    Pipeline unificado de validação: manifest × critérios biológicos × arquivo físico.
    """
    rows = []
    for rec in records:
        pid    = rec['id']
        chains = rec.get('chains', [])
        ifaces = rec.get('interfaces', [])
        prot   = [c for c in chains if c.get('mol_type') == 0]

        tamanhos  = [c['num_residues'] for c in prot] if prot else []
        validas   = [c for c in chains if c.get('valid')]
        n_res_max = max(tamanhos) if tamanhos else 0
        n_res_min = min(tamanhos) if tamanhos else 0

        tem_mhc   = n_res_max >= 200
        tem_pep   = n_res_min <= 25
        tem_2ch   = len(chains) >= 2
        if_valida = any(i.get('valid') for i in ifaces)
        tem_npz   = (structures_dir / f"{pid}.npz").exists() if structures_dir.exists() else None

        ok = tem_mhc and tem_pep and tem_2ch and if_valida
        rows.append({
            "pdb_id": pid, "n_chains": len(chains), "n_prot": len(prot),
            "max_res": n_res_max, "min_res": n_res_min,
            "tem_mhc": tem_mhc, "tem_pep": tem_pep,
            "tem_2ch": tem_2ch, "if_valida": if_valida,
            "tem_npz": tem_npz, "valido_completo": ok,
        })

    return pd.DataFrame(rows)

df_val = validacao_completa(manifest, STRUCTURES_LIMPO)

print("═" * 55)
print("  VALIDAÇÃO COMPLETA — CRITÉRIOS NEMPA")
print("═" * 55)
for col, label in [
    ("tem_2ch",        "≥ 2 cadeias"),
    ("tem_mhc",        "MHC ≥ 200 res"),
    ("tem_pep",        "Peptídeo ≤ 25 res"),
    ("if_valida",      "Interface válida"),
    ("tem_npz",        "Arquivo .npz existe"),
    ("valido_completo","Todos os critérios"),
]:
    n = df_val[col].sum() if col in df_val.columns else "N/A"
    pct = df_val[col].mean()*100 if col in df_val.columns else 0
    print(f"  {label:<25} : {n:>5}  ({pct:.1f}%)")

print("═" * 55)


### 9.2 Heatmap de Critérios

In [ ]:
criterios = [c for c in ["tem_2ch","tem_mhc","tem_pep","if_valida","tem_npz"] if c in df_val.columns]
labels = {"tem_2ch":"≥2 Cadeias", "tem_mhc":"MHC≥200",
          "tem_pep":"Pep≤25", "if_valida":"Interface
Válida", "tem_npz":".npz
Existe"}

if criterios and len(df_val) > 0:
    matriz = df_val[criterios].astype(float).fillna(-1)

    fig, ax = plt.subplots(figsize=(max(8, len(criterios)*1.5), min(30, len(df_val)*0.4+3)))
    sns.heatmap(matriz, ax=ax, cmap=["#c0392b","#f39c12","#27ae60"],
                xticklabels=[labels.get(c,c) for c in criterios],
                yticklabels=df_val['pdb_id'].str.upper(),
                linewidths=0.3, linecolor='white', cbar=False)
    ax.set_title("Heatmap de Critérios por Entrada PDB", fontsize=12)
    ax.tick_params(axis='y', labelsize=7)
    plt.tight_layout()
    plt.savefig("output/eda_13_heatmap_criterios.png", bbox_inches='tight')
    plt.show()


## <a id='fase10'></a>Fase 10 — Relatório CRISP-DM Final
### 10.1 Sumário Executivo

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║         RELATÓRIO CRISP-DM — NEMPA-Boltz (branch rafael)    ║
╠══════════════════════════════════════════════════════════════╣
║  FASE 1 — NEGÓCIO                                            ║
║    Objetivo: treinar Boltz-1 em complexos MHC-I/Peptídeo     ║
║    Critérios: MHC≥200 res, Pep≤25 res, interface válida      ║
╠══════════════════════════════════════════════════════════════╣
║  FASE 2 — ENTENDIMENTO DOS DADOS                             ║
║    Fonte: RCSB PDB (PDBx/mmCIF) via rcsb_processed_targets  ║
║    manifest.json: array de Records com extensões NEMPA       ║
║    Campos extras NEMPA: template_id, affinity, md,           ║
║                         interface.valid                       ║
╠══════════════════════════════════════════════════════════════╣
║  FASE 3 — PREPARAÇÃO DOS DADOS                               ║
║    filter_mhc.py     → filtra por PDB IDs do CSV TCR3d       ║
║    auditar_dados.py  → cruza CSV × manifest × .npz físico    ║
║    get_subsamples.py → seleciona cadeias por template JSON   ║
║    limpar_cadeias.py → aplica critérios biológicos MHC-I     ║
║    analise_descartes.py → diagnostica cada descarte          ║
║    auditoria_mascara.py / em_lote → valida arrays mask .npz  ║
╠══════════════════════════════════════════════════════════════╣
║  FASE 4-9 — MODELAGEM & AVALIAÇÃO                            ║
║    Ver outputs gerados neste notebook                        ║
╠══════════════════════════════════════════════════════════════╣
║  FASE 10 — DISTRIBUIÇÃO                                      ║
║    Dataset final: mhc_data/dataset_limpo/                    ║
║      ├── manifest.json  (índice filtrado + validado)         ║
║      ├── structures/*.npz  (mask atualizado)                 ║
║      └── msa/*.npz  (apenas MSAs de cadeias válidas)         ║
╚══════════════════════════════════════════════════════════════╝
""")


### 10.2 Geração do Relatório CSV Final

In [ ]:
# Exportar DataFrame de validação completa
if not df_val.empty:
    out_path = Path("output/nempa_validacao_completa.csv")
    df_val.to_csv(out_path, index=False)
    print(f"✅ Relatório salvo em: {out_path}")
    display(df_val.describe())

# Exportar DataFrame de chains
if not df_chains.empty:
    df_chains.to_csv("output/nempa_chains_eda.csv", index=False)
    print("✅ df_chains salvo em: output/nempa_chains_eda.csv")

# Exportar DataFrame de entradas
if not df_entries.empty:
    df_entries.to_csv("output/nempa_entries_eda.csv", index=False)
    print("✅ df_entries salvo em: output/nempa_entries_eda.csv")

print("\nArquivos gerados neste notebook:")
for f in sorted(Path("output").glob("*")):
    print(f"  📄 {f.name}  ({f.stat().st_size/1024:.1f} KB)")
